In [1]:
import magic

import numpy as np
import scanpy as sc
import pandas as pd
import os

In [ ]:
adata1 = sc.read_h5ad('human_pancreas_norm_complexBatch.h5ad')
#print(adata1.shape)
adata1

AnnData object with n_obs × n_vars = 16382 × 19093
    obs: 'tech', 'celltype', 'size_factors'
    layers: 'counts'

In [3]:
adata1.obs[:3]

,tech,celltype,size_factors
D101_5,celseq,gamma,0.028492
D101_43,celseq,gamma,0.079348
D101_93,celseq,gamma,0.037932


In [4]:
adata1.layers['counts'][:2]

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [1.0019583, 0.       , 0.       , ..., 0.       , 1.001958 ,
        0.       ]], dtype=float32)

In [5]:
adata1.X[:3]

array([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [2.6120808, 0.       , 0.       , ..., 0.       , 2.6120806,
        0.       ],
       [0.       , 3.311074 , 0.       , ..., 0.       , 0.       ,
        0.       ]], dtype=float32)

In [4]:
adata1.obs['tech'].value_counts()

tech
inDrop3       3605
smartseq2     2394
celseq2       2285
inDrop1       1937
inDrop2       1724
smarter       1492
inDrop4       1303
celseq        1004
fluidigmc1     638
Name: count, dtype: int64

In [5]:
# 确保tech列是分类类型
adata1.obs['tech'] = adata1.obs['tech'].astype('category')
# 通过映射生成新名称
adata1.obs_names = adata1.obs['tech'].astype(str) + '-' + adata1.obs_names

In [6]:
adata1.obs[:3]

,tech,celltype,size_factors
celseq-D101_5,celseq,gamma,0.028492
celseq-D101_43,celseq,gamma,0.079348
celseq-D101_93,celseq,gamma,0.037932


In [7]:
# 分层下采样：每组抽取583个细胞（不足则全取）
sampled_cells = (
    adata1.obs
    .groupby('tech', group_keys=False)
    .apply(lambda x: x.sample(n=min(583, len(x)), random_state=123))
    .index
)

# 提取子集
sampled_adata = adata1[sampled_cells, :].copy()

# 验证结果
print(sampled_adata.obs['tech'].value_counts())

tech
celseq        583
celseq2       583
fluidigmc1    583
inDrop1       583
inDrop2       583
inDrop3       583
inDrop4       583
smarter       583
smartseq2     583
Name: count, dtype: int64


In [11]:
# df = adata1.to_df().T
# print(df.shape)
# print(df.values.max())
# print(df.values.min())
# df.iloc[:3,:5]

13.002677
0.0


,celseq-D101_5,celseq-D101_43,celseq-D101_93,celseq-D102_4,celseq-D172444_23
A1BG,0.0,2.612081,0.000000,3.091586,0.000000
A1CF,0.0,0.000000,3.311074,0.000000,3.292201
A2M,0.0,0.000000,0.000000,0.000000,0.000000


In [8]:
df = sampled_adata.to_df().T
print(df.shape)
print(df.values.max())
print(df.values.min())
df.iloc[:3,:5]

(19093, 5247)
13.002677
0.0


,celseq-D1713_33,celseq-D3en2_21,celseq-D72_65,celseq-D72_67,celseq-D71_63
A1BG,0.0,0.0,0.0,0.000000,3.601441
A1CF,0.0,0.0,0.0,3.166652,0.000000
A2M,0.0,0.0,0.0,0.000000,0.000000


In [ ]:
df.to_pickle("adata_pancrease.p")

In [22]:
df_cutoff_001 = df.copy()
df_cutoff_001[:] = np.where(df_cutoff_001>=0.001, 1, 0)
df_cutoff_001.stack().value_counts()

0.0    80842284
1.0    19338687
Name: count, dtype: int64

In [12]:
# df_cutoff_001 = df.copy()
# df_cutoff_001[:] = np.where(df_cutoff_001>=0.001, 1, 0)
# df_cutoff_001.stack().value_counts()

0.0    257137681
1.0     55643845
Name: count, dtype: int64

In [9]:
annotation = pd.DataFrame({"Cell":df.columns.values, "Species":"Human"})
annotation.head()

,Cell,Species
0,celseq-D1713_33,Human
1,celseq-D3en2_21,Human
2,celseq-D72_65,Human
3,celseq-D72_67,Human
4,celseq-D71_63,Human


In [10]:
annotation['Celltype'] = annotation.Cell.apply(lambda x:x.split('-')[0])
annotation.head()

,Cell,Species,Celltype
0,celseq-D1713_33,Human,celseq
1,celseq-D3en2_21,Human,celseq
2,celseq-D72_65,Human,celseq
3,celseq-D72_67,Human,celseq
4,celseq-D71_63,Human,celseq


In [11]:
annotation['Celltype'].value_counts()

Celltype
celseq        583
celseq2       583
fluidigmc1    583
inDrop1       583
inDrop2       583
inDrop3       583
inDrop4       583
smarter       583
smartseq2     583
Name: count, dtype: int64

In [12]:
d = {'inDrop3':'inDrop','inDrop1':'inDrop','inDrop2':'inDrop','inDrop4':'inDrop','smartseq2':'smartseq2','celseq2':'celseq','celseq':'celseq',
    'smarter':'smarter','fluidigmc1':'fluidigmc1'}

In [13]:
annotation["Cluster"] = annotation["Celltype"].apply(lambda x:d[x]) 

In [14]:
annotation['Cluster'].value_counts()

Cluster
inDrop        2332
celseq        1166
fluidigmc1     583
smarter        583
smartseq2      583
Name: count, dtype: int64

In [15]:
annotation.shape

(5247, 4)

In [ ]:
annotation.to_csv("pancreas_annotation.tsv", sep='\t', index=None)

In [ ]:
df_cutoff_001.to_pickle("adata_pancrease_cutoff001.p")

In [6]:
############## 统计检测基因数目
import numpy as np
import scanpy as sc
import pandas as pd

In [ ]:
df =  pd.read_pickle("adata_pancrease.p")
print(df.shape)
df.iloc[:3,:5]

(19093, 5247)


,celseq-D1713_33,celseq-D3en2_21,celseq-D72_65,celseq-D72_67,celseq-D71_63
A1BG,0.0,0.0,0.0,0.000000,3.601441
A1CF,0.0,0.0,0.0,3.166652,0.000000
A2M,0.0,0.0,0.0,0.000000,0.000000


In [ ]:
annotation =  pd.read_csv('pancreas_annotation5247.tsv',sep='\t')
print(annotation.shape)
annotation[:2]

(5247, 4)


,Cell,Species,Celltype,Cluster
0,celseq-D1713_33,Human,celseq,celseq
1,celseq-D3en2_21,Human,celseq,celseq


In [9]:
gene_detected_PCC = np.where(df > 0, 1, 0).sum(0) 
print(gene_detected_PCC.shape)
gene_detected_AUROC = np.where(df >= 0.001, 1, 0).sum(0) 
print(gene_detected_AUROC.shape)

(5247,)
(5247,)


In [10]:
annotation["gene_detected_PCC"] = gene_detected_PCC
annotation["gene_detected_AUROC"] = gene_detected_AUROC
#annotation.head()

In [6]:
Detected_gene = annotation.groupby('Cluster')[['gene_detected_PCC','gene_detected_AUROC']].mean().round().reset_index()
Detected_gene

,Cluster,gene_detected_PCC,gene_detected_AUROC
0,celseq,4109.0,4109.0
1,fluidigmc1,7040.0,7040.0
2,inDrop,1916.0,1916.0
3,smarter,4402.0,4402.0
4,smartseq2,5848.0,5848.0


In [11]:
### MAGIC
df_magic =  pd.read_pickle("/home/ggj/Galaxy/01_benmark_data/01_run/03_BIB_pancreas_tech/PCC/MAGIC/adata_pancrease_MAGIC_PCC5245.p")
print(df_magic.shape)
#df[:3]

(19093, 5245)


In [ ]:
annotation_magic =  pd.read_csv('pancreas_annotation_MAGIC5245.tsv',sep='\t')
print(annotation_magic.shape)
annotation_magic[:2]

(5245, 4)


,Cell,Species,Celltype,Cluster
0,MAGIC-celseq-D72_65,MAGIC-Human,MAGIC-celseq,MAGIC-celseq
1,MAGIC-celseq-D72_67,MAGIC-Human,MAGIC-celseq,MAGIC-celseq


In [16]:
gene_detected_magic_PCC = np.where(df_magic > 0, 1, 0).sum(0) 
print(gene_detected_magic_PCC.shape)
gene_detected_magic_AUROC = np.where(df_magic >= 0.13, 1, 0).sum(0) 
print(gene_detected_magic_AUROC.shape)

(5245,)
(5245,)


In [17]:
annotation_magic["gene_detected_PCC"] = gene_detected_magic_PCC
annotation_magic["gene_detected_AUROC"] = gene_detected_magic_AUROC

In [11]:
Detected_gene_magic = annotation_magic.groupby('Cluster')[['gene_detected_PCC','gene_detected_AUROC']].mean().round().reset_index()
Detected_gene_magic

,Cluster,gene_detected_PCC,gene_detected_AUROC
0,MAGIC-celseq,15083.0,14309.0
1,MAGIC-fluidigmc1,15957.0,14832.0
2,MAGIC-inDrop,13882.0,13082.0
3,MAGIC-smarter,16954.0,15679.0
4,MAGIC-smartseq2,16171.0,15130.0


In [ ]:
annotation_magic.to_csv("annotation_Pancreas_magic.tsv",index=None)
annotation.to_csv("annotation_Pancreas.tsv",index=None)

In [41]:
annotation['Celltype'].unique()

array(['celseq', 'celseq2', 'fluidigmc1', 'inDrop1', 'inDrop2', 'inDrop3',
       'inDrop4', 'smarter', 'smartseq2'], dtype=object)

In [42]:
annotation['Cluster'].unique()

array(['celseq', 'fluidigmc1', 'inDrop', 'smarter', 'smartseq2'],
      dtype=object)

In [ ]:
df = pd.read_pickle("adata_pancrease.p")
annotation = pd.read_csv("pancreas_annotation5247.tsv", sep='\t')
print(df.shape)
print(annotation.shape)

(19093, 5247)
(5247, 4)


In [3]:
df_magic = []
for Tech in np.unique(annotation.Cluster.values):
    df_adata = df.iloc[:, annotation.Cluster.values==Tech].T
    
    if Tech == "BulkSeq":
        pass #df_magic.append(df_adata)
    else:
        magic_op = magic.MAGIC()
        df_adata_magic = magic_op.fit_transform(df_adata)

        print(df_adata_magic.shape)
        df_magic.append(df_adata_magic)
        
df_magic = pd.concat(df_magic, axis=0).T
df_magic = df_magic[annotation.iloc[2:,].Cell] # sort df_magic cell_i

Calculating MAGIC...
  Running MAGIC on 583 cells and 19093 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...


/home/ggj/.local/lib/python3.11/site-packages/magic_impute-3.0.0-py3.11.egg/magic/magic.py:425: UserWarning: Input matrix contains unexpressed genes. Please remove them prior to running MAGIC.
  warnings.warn(


    Calculated PCA in 0.31 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.07 seconds.
    Calculating affinities...
    Calculated affinities in 0.06 seconds.
  Calculated graph and diffusion operator in 0.46 seconds.
  Running MAGIC with `solver='exact'` on 19093-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
  Calculating imputation...
  Calculated imputation in 0.10 seconds.
Calculated MAGIC in 0.59 seconds.
(583, 19093)
Calculating MAGIC...
  Running MAGIC on 583 cells and 19093 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...


/home/ggj/.local/lib/python3.11/site-packages/magic_impute-3.0.0-py3.11.egg/magic/magic.py:425: UserWarning: Input matrix contains unexpressed genes. Please remove them prior to running MAGIC.
  warnings.warn(


    Calculated PCA in 0.27 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.06 seconds.
    Calculating affinities...
    Calculated affinities in 0.06 seconds.
  Calculated graph and diffusion operator in 0.41 seconds.
  Running MAGIC with `solver='exact'` on 19093-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
  Calculating imputation...
  Calculated imputation in 0.10 seconds.
Calculated MAGIC in 0.53 seconds.
(583, 19093)
Calculating MAGIC...
  Running MAGIC on 583 cells and 19093 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...


...

    Calculated PCA in 0.25 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.06 seconds.
    Calculating affinities...
    Calculated affinities in 0.06 seconds.
  Calculated graph and diffusion operator in 0.39 seconds.
  Running MAGIC with `solver='exact'` on 19093-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
  Calculating imputation...
  Calculated imputation in 0.08 seconds.
Calculated MAGIC in 0.49 seconds.
(583, 19093)
Calculating MAGIC...
  Running MAGIC on 583 cells and 19093 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...


/home/ggj/.local/lib/python3.11/site-packages/magic_impute-3.0.0-py3.11.egg/magic/magic.py:425: UserWarning: Input matrix contains unexpressed genes. Please remove them prior to running MAGIC.
  warnings.warn(


    Calculated PCA in 0.25 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.06 seconds.
    Calculating affinities...
    Calculated affinities in 0.06 seconds.
  Calculated graph and diffusion operator in 0.39 seconds.
  Running MAGIC with `solver='exact'` on 19093-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
  Calculating imputation...
  Calculated imputation in 0.08 seconds.
Calculated MAGIC in 0.49 seconds.
(583, 19093)
Calculating MAGIC...
  Running MAGIC on 583 cells and 19093 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...


/home/ggj/.local/lib/python3.11/site-packages/magic_impute-3.0.0-py3.11.egg/magic/magic.py:425: UserWarning: Input matrix contains unexpressed genes. Please remove them prior to running MAGIC.
  warnings.warn(


    Calculated PCA in 0.25 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.06 seconds.
    Calculating affinities...
    Calculated affinities in 0.06 seconds.
  Calculated graph and diffusion operator in 0.39 seconds.
  Running MAGIC with `solver='exact'` on 19093-dimensional data may take a long time. Consider denoising specific genes with `genes=<list-like>` or using `solver='approximate'`.
  Calculating imputation...
  Calculated imputation in 0.08 seconds.
Calculated MAGIC in 0.49 seconds.
(583, 19093)


In [5]:
df_magic[:2]

,celseq-D72_65,celseq-D72_67,celseq-D71_63,celseq-D3en2_26,celseq-D2ex_17,celseq-D71_62,celseq-D3en4_71,celseq-D17TGFB_19,celseq-D101_10,celseq-D71_78,...,smartseq2-HP1506401_G18,smartseq2-HP1525301T2D_O2,smartseq2-HP1508501T2D_A6,smartseq2-HP1506401_N5,smartseq2-HP1504901_F10,smartseq2-HP1504101T2D_N8,smartseq2-HP1507101_C8,smartseq2-HP1508501T2D_J14,smartseq2-HP1502401_C1,smartseq2-HP1508501T2D_K3
A1BG,0.054418,1.644973,1.925095,0.216995,0.147397,1.758142,1.668220,0.292099,1.642259,0.445424,...,0.113453,1.099451,1.510646,0.138682,1.376218,1.893660,2.216099,0.064688,0.136796,0.066099
A1CF,0.054921,1.637339,0.568617,0.740218,0.458273,0.627715,1.627917,0.229016,1.546143,0.856088,...,0.162872,1.371866,1.599284,2.133069,2.258712,2.463081,2.333304,0.073120,0.947530,0.088577


In [6]:
print(df_magic.values.max())
print(df_magic.values.min())
print(df_magic.shape)

11.366844635241916
0.0
(19093, 5245)


In [57]:
df[:3]

,celseq-D1713_33,celseq-D3en2_21,celseq-D72_65,celseq-D72_67,celseq-D71_63,celseq-D3en2_26,celseq-D2ex_17,celseq-D71_62,celseq-D3en4_71,celseq-D17TGFB_19,...,smartseq2-HP1506401_G18,smartseq2-HP1525301T2D_O2,smartseq2-HP1508501T2D_A6,smartseq2-HP1506401_N5,smartseq2-HP1504901_F10,smartseq2-HP1504101T2D_N8,smartseq2-HP1507101_C8,smartseq2-HP1508501T2D_J14,smartseq2-HP1502401_C1,smartseq2-HP1508501T2D_K3
A1BG,0.0,0.0,0.0,0.000000,3.601441,0.0,0.0,3.106612,3.08432,0.0,...,0.00000,0.0,0.0,0.000000,3.171343,2.125821,3.214507,0.000000,0.000000,0.0
A1CF,0.0,0.0,0.0,3.166652,0.000000,0.0,0.0,0.000000,3.08432,0.0,...,0.00000,0.0,0.0,2.994655,4.407980,4.136567,2.699494,0.000000,0.305924,0.0
A2M,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.00000,0.0,...,0.05754,0.0,0.0,1.691628,0.000000,0.000000,0.000000,0.351045,0.000000,0.0


In [11]:
annotation_magic = pd.DataFrame({"Cell":df_magic.columns.values, "Species":"Human"})
annotation_magic['Celltype'] = annotation_magic.Cell.apply(lambda x:x.split('-')[0])

In [12]:
d = {'inDrop3':'inDrop','inDrop1':'inDrop','inDrop2':'inDrop','inDrop4':'inDrop','smartseq2':'smartseq2','celseq2':'celseq','celseq':'celseq',
    'smarter':'smarter','fluidigmc1':'fluidigmc1'}
annotation_magic["Cluster"] = annotation_magic["Celltype"].apply(lambda x:d[x]) 
annotation_magic.head()

,Cell,Species,Celltype,Cluster
0,celseq-D72_65,Human,celseq,celseq
1,celseq-D72_67,Human,celseq,celseq
2,celseq-D71_63,Human,celseq,celseq
3,celseq-D3en2_26,Human,celseq,celseq
4,celseq-D2ex_17,Human,celseq,celseq


In [64]:
#annotation2 = annotation[annotation['Cell'].isin(df_magic.columns)]
#print(annotation2.shape)

(5245, 4)


In [14]:
#annotation_magic = annotation2.copy()
annotation2 = annotation_magic.copy()
annotation_magic.Cell = annotation2.Cell.apply(lambda x:"MAGIC-"+x)
annotation_magic.Species = annotation2.Species.apply(lambda x:"MAGIC-"+x)
annotation_magic.Celltype = annotation2.Celltype.apply(lambda x:"MAGIC-"+x)
annotation_magic.Cluster = annotation2.Cluster.apply(lambda x:"MAGIC-"+x)
print(annotation_magic.shape)
annotation_magic.head()

(5245, 4)


,Cell,Species,Celltype,Cluster
0,MAGIC-celseq-D72_65,MAGIC-Human,MAGIC-celseq,MAGIC-celseq
1,MAGIC-celseq-D72_67,MAGIC-Human,MAGIC-celseq,MAGIC-celseq
2,MAGIC-celseq-D71_63,MAGIC-Human,MAGIC-celseq,MAGIC-celseq
3,MAGIC-celseq-D3en2_26,MAGIC-Human,MAGIC-celseq,MAGIC-celseq
4,MAGIC-celseq-D2ex_17,MAGIC-Human,MAGIC-celseq,MAGIC-celseq


In [63]:
#annotation[~annotation['Cell'].isin(df_magic.columns)]

,Cell,Species,Celltype,Cluster
0,celseq-D1713_33,Human,celseq,celseq
1,celseq-D3en2_21,Human,celseq,celseq


In [15]:
df_magic.columns = annotation_magic.Cell
df_magic[:2]

Cell,MAGIC-celseq-D72_65,MAGIC-celseq-D72_67,MAGIC-celseq-D71_63,MAGIC-celseq-D3en2_26,MAGIC-celseq-D2ex_17,MAGIC-celseq-D71_62,MAGIC-celseq-D3en4_71,MAGIC-celseq-D17TGFB_19,MAGIC-celseq-D101_10,MAGIC-celseq-D71_78,...,MAGIC-smartseq2-HP1506401_G18,MAGIC-smartseq2-HP1525301T2D_O2,MAGIC-smartseq2-HP1508501T2D_A6,MAGIC-smartseq2-HP1506401_N5,MAGIC-smartseq2-HP1504901_F10,MAGIC-smartseq2-HP1504101T2D_N8,MAGIC-smartseq2-HP1507101_C8,MAGIC-smartseq2-HP1508501T2D_J14,MAGIC-smartseq2-HP1502401_C1,MAGIC-smartseq2-HP1508501T2D_K3
A1BG,0.054418,1.644973,1.925095,0.216995,0.147397,1.758142,1.668220,0.292099,1.642259,0.445424,...,0.113453,1.099451,1.510646,0.138682,1.376218,1.893660,2.216099,0.064688,0.136796,0.066099
A1CF,0.054921,1.637339,0.568617,0.740218,0.458273,0.627715,1.627917,0.229016,1.546143,0.856088,...,0.162872,1.371866,1.599284,2.133069,2.258712,2.463081,2.333304,0.073120,0.947530,0.088577


In [16]:
df_magic.columns.name = None
df_magic[:2]

,MAGIC-celseq-D72_65,MAGIC-celseq-D72_67,MAGIC-celseq-D71_63,MAGIC-celseq-D3en2_26,MAGIC-celseq-D2ex_17,MAGIC-celseq-D71_62,MAGIC-celseq-D3en4_71,MAGIC-celseq-D17TGFB_19,MAGIC-celseq-D101_10,MAGIC-celseq-D71_78,...,MAGIC-smartseq2-HP1506401_G18,MAGIC-smartseq2-HP1525301T2D_O2,MAGIC-smartseq2-HP1508501T2D_A6,MAGIC-smartseq2-HP1506401_N5,MAGIC-smartseq2-HP1504901_F10,MAGIC-smartseq2-HP1504101T2D_N8,MAGIC-smartseq2-HP1507101_C8,MAGIC-smartseq2-HP1508501T2D_J14,MAGIC-smartseq2-HP1502401_C1,MAGIC-smartseq2-HP1508501T2D_K3
A1BG,0.054418,1.644973,1.925095,0.216995,0.147397,1.758142,1.668220,0.292099,1.642259,0.445424,...,0.113453,1.099451,1.510646,0.138682,1.376218,1.893660,2.216099,0.064688,0.136796,0.066099
A1CF,0.054921,1.637339,0.568617,0.740218,0.458273,0.627715,1.627917,0.229016,1.546143,0.856088,...,0.162872,1.371866,1.599284,2.133069,2.258712,2.463081,2.333304,0.073120,0.947530,0.088577


In [ ]:
annotation_magic.to_csv("pancreas_annotation_MAGIC5245.tsv", sep='\t', index=None)

In [ ]:
df_magic.to_pickle("./MAGIC/adata_pancrease_MAGIC_PCC5245.p")

In [68]:
df_magic_cutoff_001 = df_magic.copy()
df_magic_cutoff_001[:] = np.where(df_magic_cutoff_001 >= 0.001, 1, 0)
df_magic_cutoff_001.stack().value_counts()

1.0    73774930
0.0    26367855
Name: count, dtype: int64

In [ ]:
#annotation_magic.to_csv("/home/ggj/virtual/bib/01_run/03_BIB_pancreas_tech/MAGIC/pancreas_annotation_MAGIC.tsv", sep='\t', index=None)
df_magic_cutoff_001.to_pickle("./MAGIC/adata_pancrease_MAGIC_cutoff001.p")

In [ ]:
####################### 将中位值作为cutoff值

In [12]:
import numpy as np
import scanpy as sc
import pandas as pd

In [ ]:
df =  pd.read_pickle("adata_pancrease.p")
print(df.shape)
df.iloc[:3,:5]

(19093, 5247)


,celseq-D1713_33,celseq-D3en2_21,celseq-D72_65,celseq-D72_67,celseq-D71_63
A1BG,0.0,0.0,0.0,0.000000,3.601441
A1CF,0.0,0.0,0.0,3.166652,0.000000
A2M,0.0,0.0,0.0,0.000000,0.000000


In [35]:
print(df.values.max())
print(np.median(df.values)) 
print(df.values.min())

13.002677
0.0
0.0


In [ ]:
df_magic =  pd.read_pickle("./MAGIC/adata_pancrease_MAGIC_PCC5245.p")

In [15]:
df_magic.iloc[:3,:5]

,MAGIC-celseq-D72_65,MAGIC-celseq-D72_67,MAGIC-celseq-D71_63,MAGIC-celseq-D3en2_26,MAGIC-celseq-D2ex_17
A1BG,0.054418,1.644973,1.925095,0.216995,0.147397
A1CF,0.054921,1.637339,0.568617,0.740218,0.458273
A2M,0.052931,0.035023,0.002081,0.050109,0.022382


In [27]:
print(df_magic.values.max())
print(df_magic.values.mean())
#print(df_magic.values.median())
print(np.median(df_magic.values)) 
print(df_magic.values.min())

print(df_magic.shape)

11.366844635241916
0.5228266721337372
0.13406992218923025
0.0
(19093, 5245)


In [28]:
t=0.13
df_magic_cutoff_t = df_magic.copy()
df_magic_cutoff_t[:] = np.where(df_magic_cutoff_t >= t, 1, 0)
df_magic_cutoff_t.stack().value_counts()

1.0    50402917
0.0    49739868
Name: count, dtype: int64

In [32]:
print(np.median(df_magic.values)) 

0.13406992218923025


In [36]:
df_magic_cutoff_13 = df_magic.copy()
df_magic_cutoff_13[:] = np.where(df_magic_cutoff_13 >= 0.13, 1, 0)
df_magic_cutoff_13.stack().value_counts()

1.0    50402917
0.0    49739868
Name: count, dtype: int64

In [ ]:
df_magic_cutoff_13.to_pickle("./MAGIC/adata_pancrease_MAGIC_cutoff13.p")

In [31]:
df_magic_cutoff_13[:3]

,MAGIC-celseq-D72_65,MAGIC-celseq-D72_67,MAGIC-celseq-D71_63,MAGIC-celseq-D3en2_26,MAGIC-celseq-D2ex_17,MAGIC-celseq-D71_62,MAGIC-celseq-D3en4_71,MAGIC-celseq-D17TGFB_19,MAGIC-celseq-D101_10,MAGIC-celseq-D71_78,...,MAGIC-smartseq2-HP1506401_G18,MAGIC-smartseq2-HP1525301T2D_O2,MAGIC-smartseq2-HP1508501T2D_A6,MAGIC-smartseq2-HP1506401_N5,MAGIC-smartseq2-HP1504901_F10,MAGIC-smartseq2-HP1504101T2D_N8,MAGIC-smartseq2-HP1507101_C8,MAGIC-smartseq2-HP1508501T2D_J14,MAGIC-smartseq2-HP1502401_C1,MAGIC-smartseq2-HP1508501T2D_K3
A1BG,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
A1CF,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
A2M,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [18]:
annotation_magic['Celltype'].value_counts()

Celltype
MAGIC-celseq2       583
MAGIC-inDrop3       583
MAGIC-fluidigmc1    583
MAGIC-inDrop1       583
MAGIC-inDrop2       583
MAGIC-smarter       583
MAGIC-inDrop4       583
MAGIC-smartseq2     583
MAGIC-celseq        581
Name: count, dtype: int64

In [19]:
annotation_magic['Cluster'].value_counts()

Cluster
MAGIC-inDrop        2332
MAGIC-celseq        1164
MAGIC-fluidigmc1     583
MAGIC-smarter        583
MAGIC-smartseq2      583
Name: count, dtype: int64